## Supuestos económicos

Se definen los siguientes supuestos, documentados explícitamente para que puedan
cuestionarse o ajustarse (mismo criterio usado en la Etapa 1):

- **Costo de adquisición de cliente nuevo**: 5x el cargo mensual promedio
- **Costo de campaña de retención**: 15% del costo de adquisición
- **LTV restante**: Monthly Charges × meses restantes estimados (a calcular en el paso 2)
- **Tasa de éxito de campaña de retención**: 35%

Estos valores son estimaciones razonadas, no datos exactos de Finanzas — el análisis
de sensibilidad del paso 7 va a mostrar qué tan robusta es la conclusión ante cambios
en estos supuestos.

In [2]:
import pandas as pd

X = pd.read_csv('../data/processed/X_features.csv')
y = pd.read_csv('../data/processed/y_target.csv')

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

X_train = X_train.drop(columns=['Grupo_Antiguedad'])
X_test = X_test.drop(columns=['Grupo_Antiguedad'])

print(X_train.shape)
print(X_test.shape)

(5634, 28)
(1409, 28)


In [4]:
MULTIPLICADOR_COSTO_ADQUISICION = 5
PORCENTAJE_COSTO_CAMPANA = 0.15
TASA_EXITO_CAMPANA = 0.35

cargo_mensual_promedio = X_train['Monthly Charges'].mean()
costo_adquisicion = cargo_mensual_promedio * MULTIPLICADOR_COSTO_ADQUISICION
costo_campana = costo_adquisicion * PORCENTAJE_COSTO_CAMPANA

print(f"Cargo mensual promedio: ${cargo_mensual_promedio:.2f}")
print(f"Costo de adquisición: ${costo_adquisicion:.2f}")
print(f"Costo de campaña de retención: ${costo_campana:.2f}")

Cargo mensual promedio: $64.93
Costo de adquisición: $324.65
Costo de campaña de retención: $48.70


### Resultado: costos base definidos

Con un cargo mensual promedio de $64.93, el costo de adquisición estimado es de
$324.65, y el costo de campaña de retención de $48.70. Estos valores se usan como
base para el cálculo de LTV y el ahorro neto en los pasos siguientes.

In [5]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(5634, 29)
(1409, 29)
(5634, 1)
(1409, 1)


## Calcular el LTV (Lifetime Value) por cliente

Se estima el LTV de cada cliente como `Monthly Charges × meses restantes estimados`.
Como no se puede saber con certeza cuántos meses más seguirá un cliente, se aproxima
usando el promedio de antigüedad de los clientes que NO se fueron (Churn Value = 0),
como una estimación razonable de cuánto dura típicamente una relación de largo plazo
con la empresa.

In [7]:
meses_restantes_estimados = X_train[y_train['Churn Value'] == 0]['Tenure Months'].mean()
print(f"Meses restantes estimados (promedio): {meses_restantes_estimados:.1f}")

Meses restantes estimados (promedio): 37.6


### Resultado: meses restantes estimados

37.6 meses en promedio — se usa como estimación de cuánto tiempo más permanecería un
cliente si se retiene exitosamente, basado en el comportamiento real de los clientes
que no se fueron.

## Cálculo de LTV por cliente

Se calcula el LTV individual de cada cliente en X_test (el set que vamos a usar para
las matrices de confusión), multiplicando su cargo mensual actual por los 37.6 meses
restantes estimados. Esto da una estimación personalizada, no un promedio general
único para todos los clientes.

In [8]:
LTV = X_test['Monthly Charges'] * meses_restantes_estimados

print(f"LTV promedio: ${LTV.mean():.2f}")
print(f"LTV mínimo: ${LTV.min():.2f}")
print(f"LTV máximo: ${LTV.max():.2f}")

LTV promedio: $2408.95
LTV mínimo: $685.97
LTV máximo: $4390.23


### Resultado: LTV por cliente

LTV promedio: $2,408.95 | Mínimo: $685.97 | Máximo: $4,390.23

El rango refleja la variabilidad real de cargos mensuales entre clientes (desde
servicios básicos hasta combos completos), multiplicado por la misma estimación de
37.6 meses restantes para todos. Este LTV individual se usa en el paso 3 para
traducir cada Verdadero Positivo y Falso Negativo de la matriz de confusión a un
valor económico concreto.

## Comparación con CLTV de IBM

Se filtran del dataset original (`df`, con las 33 columnas) los mismos 1,409 clientes
que componen X_test, usando el índice compartido entre ambos. Se compara el LTV propio
contra la columna CLTV ya calculada por IBM, para contrastar ambos criterios.

In [11]:
df = pd.read_excel('../data/raw/Telco_customer_churn.xlsx', sheet_name='Telco_Churn')
df.shape

(7043, 33)

In [12]:
cltv_ibm = df.loc[X_test.index, 'CLTV']

print(f"LTV propio - promedio: ${LTV.mean():.2f}")
print(f"CLTV de IBM - promedio: ${cltv_ibm.mean():.2f}")

LTV propio - promedio: $2408.95
CLTV de IBM - promedio: $4385.08


### Resultado: comparación LTV propio vs. CLTV de IBM

LTV propio (promedio): $2,408.95 | CLTV de IBM (promedio): $4,385.08

El CLTV de IBM casi duplica la estimación propia. La diferencia probablemente se debe
a una metodología más sofisticada de parte de IBM (no documentada en el dataset —
posiblemente incorpora un horizonte de tiempo más largo, proyecciones de upselling, o
descuento financiero) frente a la fórmula simple y transparente usada acá (cargo
mensual actual × antigüedad promedio de clientes retenidos).

**Decisión**: se continúa usando el LTV propio para el cálculo de ahorro neto, por ser
completamente auditable y basado en supuestos explícitos, documentados y ajustables —
en línea con el criterio de transparencia usado en todo el proyecto. El CLTV de IBM
queda documentado como referencia de contraste, no como reemplazo.

## Traducir la matriz de confusión a pesos

Se calcula el ahorro neto de cada modelo aplicando la fórmula económica definida en
el planning original:

- **Verdadero Positivo (VP)**: Beneficio = (LTV del cliente × 0.35) − $48.70
- **Falso Positivo (FP)**: Beneficio = −$48.70

Los Falsos Negativos y Verdaderos Negativos no participan directamente en la fórmula,
ya que en ambos escenarios (con o sin modelo) su destino es el mismo: los FN se van
igual (nadie los contactó) y los VN se quedan igual (correctamente, sin gasto). El
valor del modelo está en a quién decide contactar (VP + FP).

A diferencia de usar un LTV promedio general, se identifica a los clientes específicos
que fueron VP para sumar su LTV individual (calculado en el paso 2), preservando la
variabilidad real entre clientes de mayor y menor cargo mensual.

### Preparación: reentrenar los 2 modelos finalistas

Se reentrenan Gradient Boosting y Regresión Logística en este notebook (mismo proceso
que en la Etapa 4), para generar las predicciones y probabilidades sobre X_test
necesarias para el cálculo de impacto económico.

In [13]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

modelo_gb = GradientBoostingClassifier(random_state=42)
modelo_gb.fit(X_train, y_train['Churn Value'])
predicciones_gb = modelo_gb.predict(X_test)
probabilidades_gb = modelo_gb.predict_proba(X_test)[:, 1]

modelo_escalado = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
])
modelo_escalado.fit(X_train, y_train['Churn Value'])
predicciones_lr = modelo_escalado.predict(X_test)
probabilidades_lr = modelo_escalado.predict_proba(X_test)[:, 1]

print("Modelos reentrenados correctamente")

Modelos reentrenados correctamente


In [14]:
es_vp = (predicciones_gb == 1) & (y_test['Churn Value'] == 1)
es_fp = (predicciones_gb == 1) & (y_test['Churn Value'] == 0)

print(f"Cantidad de VP: {es_vp.sum()}")
print(f"Cantidad de FP: {es_fp.sum()}")

Cantidad de VP: 196
Cantidad de FP: 107


#### Resultado: identificación de VP y FP (Gradient Boosting)

VP: 196 | FP: 107 — coincide exactamente con la matriz de confusión calculada en la
Etapa 4, confirmando consistencia entre notebooks.

### Cálculo de ahorro neto — Gradient Boosting

Se aplica la fórmula económica: para los 196 VP, se suma su LTV individual multiplicado
por la tasa de éxito de campaña (35%), restando el costo de campaña; para los 107 FP,
se resta directamente el costo de campaña. La suma de ambos componentes da el ahorro
neto total de aplicar el modelo.

In [15]:
LTV_de_los_vp = LTV[es_vp]

beneficio_vp = (LTV_de_los_vp * TASA_EXITO_CAMPANA - costo_campana).sum()
costo_fp = es_fp.sum() * costo_campana

ahorro_neto_gb = beneficio_vp - costo_fp

print(f"Beneficio por VP: ${beneficio_vp:.2f}")
print(f"Costo por FP: ${costo_fp:.2f}")
print(f"Ahorro neto total (Gradient Boosting): ${ahorro_neto_gb:.2f}")

Beneficio por VP: $191867.69
Costo por FP: $5210.63
Ahorro neto total (Gradient Boosting): $186657.06


#### Resultado: ahorro neto de Gradient Boosting

Beneficio por VP: $191,867.69 | Costo por FP: $5,210.63 | **Ahorro neto: $186,657.06**

El modelo generaría un ahorro estimado de $186,657 sobre los 1,409 clientes del set de
test. El costo de los Falsos Positivos representa solo el 2.7% del beneficio generado
por los Verdaderos Positivos, sugiriendo que el modelo aporta valor económico neto
incluso sin ser perfectamente preciso.

### Cálculo de ahorro neto — Regresión Logística

Se repite el mismo cálculo con las predicciones de Regresión Logística, para comparar
directamente contra Gradient Boosting con los mismos supuestos económicos.

In [16]:
es_vp_lr = (predicciones_lr == 1) & (y_test['Churn Value'] == 1)
es_fp_lr = (predicciones_lr == 1) & (y_test['Churn Value'] == 0)

LTV_de_los_vp_lr = LTV[es_vp_lr]

beneficio_vp_lr = (LTV_de_los_vp_lr * TASA_EXITO_CAMPANA - costo_campana).sum()
costo_fp_lr = es_fp_lr.sum() * costo_campana

ahorro_neto_lr = beneficio_vp_lr - costo_fp_lr

print(f"Cantidad de VP: {es_vp_lr.sum()}")
print(f"Cantidad de FP: {es_fp_lr.sum()}")
print(f"Beneficio por VP: ${beneficio_vp_lr:.2f}")
print(f"Costo por FP: ${costo_fp_lr:.2f}")
print(f"Ahorro neto total (Regresión Logística): ${ahorro_neto_lr:.2f}")

Cantidad de VP: 293
Cantidad de FP: 281
Beneficio por VP: $278398.89
Costo por FP: $13683.99
Ahorro neto total (Regresión Logística): $264714.90


#### Resultado: ahorro neto de Regresión Logística

Beneficio por VP: $278,398.89 | Costo por FP: $13,683.99 | **Ahorro neto: $264,714.90**

Regresión Logística genera un ahorro neto 42% mayor que Gradient Boosting ($264,715 vs
$186,657), a pesar de tener casi el triple de Falsos Positivos (281 vs 107). Esto se
explica por la relación entre los supuestos económicos: el costo de campaña ($48.70)
es bajo comparado con el beneficio potencial de retener un cliente (~$843 en promedio),
por lo que priorizar Recall (detectar más clientes reales) resulta más rentable que
priorizar Precision, con estos supuestos específicos.

## Comparación de los 3 escenarios de negocio

### Escenario 1: No hacer nada

Se calcula el costo de no actuar sobre ningún cliente: se pierde el LTV completo de
todos los que realmente se van, sin gasto en campañas (porque no se contacta a nadie).

In [17]:
son_churn_real = y_test['Churn Value'] == 1
LTV_de_los_que_se_van = LTV[son_churn_real]

costo_escenario_1 = LTV_de_los_que_se_van.sum()

print(f"Cantidad de clientes que realmente se van: {son_churn_real.sum()}")
print(f"Costo de no hacer nada: ${costo_escenario_1:.2f}")

Cantidad de clientes que realmente se van: 374
Costo de no hacer nada: $1022942.27


### Resultado: Escenario 1 (No hacer nada)

374 clientes se van sin intervención. Costo total: **$1,022,942.27** (LTV completo
perdido, sin ningún gasto en campañas). Este es el punto de referencia contra el que
se miden los otros dos escenarios.

### Escenario 2: Campaña masiva a ciegas

Se calcula el resultado de ofrecer la campaña de retención a los 1,409 clientes de
test, sin usar ningún modelo. Los que realmente se iban a ir tienen 35% de probabilidad
de ser retenidos (mismo supuesto de siempre); el resto es puro gasto de campaña sin
ningún beneficio.

In [18]:
beneficio_retenidos = (LTV_de_los_que_se_van * TASA_EXITO_CAMPANA - costo_campana).sum()
costo_resto = (len(X_test) - son_churn_real.sum()) * costo_campana

ahorro_neto_escenario_2 = beneficio_retenidos - costo_resto

print(f"Beneficio de los que se iban a ir (374): ${beneficio_retenidos:.2f}")
print(f"Costo de campaña al resto ({len(X_test) - son_churn_real.sum()} clientes): ${costo_resto:.2f}")
print(f"Ahorro neto total (campaña masiva): ${ahorro_neto_escenario_2:.2f}")

Beneficio de los que se iban a ir (374): $339816.94
Costo de campaña al resto (1035 clientes): $50401.88
Ahorro neto total (campaña masiva): $289415.06


### Resultado: Escenario 2 (Campaña masiva)

Beneficio de los que se iban a ir: $339,816.94 | Costo de campaña al resto (1,035
clientes): $50,401.88 | **Ahorro neto: $289,415.06**

**Hallazgo importante**: con los supuestos económicos actuales, la campaña masiva a
ciegas genera un ahorro neto MAYOR que ambos modelos (Regresión Logística: $264,715 |
Gradient Boosting: $186,657). Esto se debe a que el costo de campaña ($48.70) es muy
bajo en relación al beneficio potencial de retención (~$843 por cliente), haciendo
rentable contactar indiscriminadamente. Este resultado no invalida el valor del
modelo — sugiere que la fórmula actual no captura costos reales de una campaña masiva
que no son perfectamente lineales (saturación de contacto, desgaste de marca, límites
operativos de escala). Se profundiza este punto en el análisis de sensibilidad
(paso 7) y en las conclusiones finales.

### Resumen: los 3 escenarios comparados

| Escenario | Ahorro neto |
|---|---|
| 1. No hacer nada | -$1,022,942.27 |
| 2. Campaña masiva a ciegas | $289,415.06 |
| 3a. Modelo Gradient Boosting | $186,657.06 |
| 3b. Modelo Regresión Logística | $264,714.90 |

Los tres escenarios de acción (2, 3a, 3b) superan ampliamente a no hacer nada. Entre
los escenarios de acción, la campaña masiva resulta la de mayor ahorro con los
supuestos actuales — un resultado a analizar en profundidad con el ajuste de umbral
(paso 5) y el análisis de sensibilidad (paso 7).

## Optimización del umbral de decisión

Se prueba primero la función `calcular_ahorro_neto()` con un solo umbral (0.5), para
confirmar que reproduce exactamente el mismo resultado ya calculado manualmente en el
paso 3, antes de recorrer múltiples umbrales.

In [21]:
import sys
sys.path.append('..')

from src.economics import calcular_ahorro_neto

In [22]:
ahorro_prueba = calcular_ahorro_neto(
    probabilidades_gb, 
    y_test['Churn Value'], 
    LTV, 
    umbral=0.5, 
    tasa_exito=TASA_EXITO_CAMPANA, 
    costo_campana=costo_campana
)

print(f"Ahorro neto con umbral 0.5: ${ahorro_prueba:.2f}")

Ahorro neto con umbral 0.5: $186657.06


### Recorrido de umbrales — Gradient Boosting

Se recorre un rango de umbrales de 0.05 a 0.95 (en pasos de 0.05), calculando el
ahorro neto para cada uno con la función `calcular_ahorro_neto()`. El objetivo es
encontrar el umbral que maximiza el ahorro neto, en vez de usar el 0.5 por default.

In [23]:
import numpy as np

umbrales = np.arange(0.05, 1.0, 0.05)

resultados_umbrales_gb = []

for u in umbrales:
    ahorro = calcular_ahorro_neto(probabilidades_gb, y_test['Churn Value'], LTV, u, TASA_EXITO_CAMPANA, costo_campana)
    resultados_umbrales_gb.append(ahorro)

tabla_umbrales_gb = pd.DataFrame({
    'Umbral': umbrales,
    'Ahorro Neto': resultados_umbrales_gb
})

tabla_umbrales_gb

,Umbral,Ahorro Neto
0,0.05,306922.618570
1,0.10,300950.154474
2,0.15,289460.673948
3,0.20,283351.974687
4,0.25,270390.581707
5,0.30,257036.212286
6,0.35,243032.546704
7,0.40,226303.903018
8,0.45,206002.669193
9,0.50,186657.064281


### Ajuste: extender el rango de umbrales

El ahorro neto seguía creciendo hasta el umbral más bajo probado (0.05), sin mostrar
un máximo claro. Se extiende el rango hacia abajo (desde 0.01) para confirmar si
existe un verdadero punto óptimo, o si el ahorro neto sigue aumentando monótonamente
a medida que el umbral baja.

In [26]:
umbrales = np.arange(0.01, 1.0, 0.02)

resultados_umbrales_gb = []

for u in umbrales:
    ahorro = calcular_ahorro_neto(probabilidades_gb, y_test['Churn Value'], LTV, u, TASA_EXITO_CAMPANA, costo_campana)
    resultados_umbrales_gb.append(ahorro)

tabla_umbrales_gb = pd.DataFrame({
    'Umbral': umbrales,
    'Ahorro Neto': resultados_umbrales_gb
})

tabla_umbrales_gb.head(10)

,Umbral,Ahorro Neto
0,0.01,292872.579698
1,0.03,301639.321024
2,0.05,306922.618570
3,0.07,304729.915111
4,0.09,302863.105466
5,0.11,299417.251001
6,0.13,295966.554978
7,0.15,289460.673948
8,0.17,288491.662650
9,0.19,285870.409031


#### Resultado: umbral óptimo de Gradient Boosting

El ahorro neto alcanza su máximo en el **umbral 0.05**, con **$306,922.62** — superando
tanto al umbral por default (0.5 → $186,657.06) como a la campaña masiva a ciegas
($289,415.06). Confirmado como verdadero máximo: el ahorro sube desde umbrales muy
bajos hasta 0.05, y decrece consistentemente a partir de ahí.

### Recorrido de umbrales — Regresión Logística

Se repite el mismo proceso con las probabilidades de Regresión Logística, para
encontrar su propio umbral óptimo y compararlo contra Gradient Boosting.

In [28]:
umbrales = np.arange(0.01, 1.0, 0.02)

resultados_umbrales_lr = []

for u in umbrales:
    ahorro = calcular_ahorro_neto(probabilidades_lr, y_test['Churn Value'], LTV, u, TASA_EXITO_CAMPANA, costo_campana)
    resultados_umbrales_lr.append(ahorro)

tabla_umbrales_lr = pd.DataFrame({
    'Umbral': umbrales,
    'Ahorro Neto': resultados_umbrales_lr
})

tabla_umbrales_lr.head(15)

,Umbral,Ahorro Neto
0,0.01,292677.789815
1,0.03,297180.008791
2,0.05,299177.801688
3,0.07,301129.312719
4,0.09,303111.124859
5,0.11,304790.648355
6,0.13,306105.480064
7,0.15,306177.630981
8,0.17,305525.207124
9,0.19,304341.057786


#### Resultado: umbral óptimo de Regresión Logística

El ahorro neto alcanza su máximo cerca del **umbral 0.15**, con **$306,177.63**.
Comparado con Gradient Boosting en su propio óptimo (umbral 0.05, $306,922.62), la
diferencia es mínima (~$745) — ambos modelos, una vez optimizado su umbral, generan
un ahorro prácticamente idéntico y superan cómodamente tanto al umbral default (0.5)
como a la campaña masiva a ciegas ($289,415.06).

## Gráfico: Ahorro Neto vs. Umbral

Se grafican las curvas completas de ahorro neto para cada umbral, comparando ambos
modelos en un solo gráfico. Se incluye una línea horizontal de referencia con el
ahorro neto de la campaña masiva, para visualizar en qué rango de umbrales cada
modelo logra superarla.

In [30]:
import plotly.express as px

tabla_umbrales_gb['Modelo'] = 'Gradient Boosting'
tabla_umbrales_lr['Modelo'] = 'Regresión Logística'

comparacion_umbrales = pd.concat([tabla_umbrales_gb, tabla_umbrales_lr])

fig = px.line(
    comparacion_umbrales,
    x='Umbral',
    y='Ahorro Neto',
    color='Modelo',
    title='Ahorro Neto vs. Umbral de Decisión'
)

fig.add_hline(y=ahorro_neto_escenario_2, line_dash='dash', annotation_text='Campaña masiva a ciegas')

fig.show()

### Resultado: gráfico Ahorro Neto vs. Umbral

Ambos modelos alcanzan un pico similar en su ahorro neto óptimo (~$306,000), pero con
formas de curva muy distintas: Gradient Boosting tiene un pico angosto (óptimo cerca
de 0.05, cayendo rápido fuera de ese rango), mientras que Regresión Logística mantiene
una meseta amplia y estable entre 0.01 y ~0.3, superando a la campaña masiva en una
ventana de umbrales mucho más ancha. Esto sugiere que Regresión Logística es más
robusta ante variaciones en el umbral elegido — una ventaja práctica relevante para
la implementación en producción, más allá del valor máximo puntual.